# 2. Generate AEDP Different-Aggregation Datasets

Heavy notebook for manual execution after notebook 1. It creates 40 weather-enhanced PyNNLF datasets.

## 1. Setup And Paths

In [ ]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
import numpy as np
PROCESSED_DIR = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\2. processed")
CLEANED_WORKSPACE = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation")
SITE_LIST_PATH = PROCESSED_DIR / "aedp_cluster_2_2_3years.xlsx"
WEATHER_PATH = PROCESSED_DIR / "aedp_weather_data.csv"
SITE_30MIN_DIR = CLEANED_WORKSPACE / "checkpoints" / "site_30min"
AUDIT_DIR = CLEANED_WORKSPACE / "audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DATA_DIR = REPO_ROOT / "data"
PUBLICATION_DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "03_aedp_aggregation_level"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
START = pd.Timestamp("2021-07-01 00:00:00")
END_30MIN = pd.Timestamp("2024-06-30 23:30:00")
INDEX_30MIN = pd.date_range(START, END_30MIN, freq="30min", name="datetime")
RANDOM_SEED = 20260521
SAMPLES_PER_LEVEL = 10
AGGREGATION_LEVELS = [1, 10, 100, 1000]
for path in [SITE_LIST_PATH, WEATHER_PATH, SITE_30MIN_DIR, ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Cleaned AEDP workspace: {CLEANED_WORKSPACE}")

## 2. Load Site List And Weather

In [ ]:
site_list = pd.read_excel(SITE_LIST_PATH)
site_list["edp_site_id"] = site_list["edp_site_id"].astype(str)
site_ids = site_list["edp_site_id"].tolist()
if len(site_ids) != 148 or len(set(site_ids)) != 148:
    raise ValueError("Expected 148 unique AEDP site IDs")
missing_checkpoints = [site_id for site_id in site_ids if not (SITE_30MIN_DIR / f"{site_id}_30min.parquet").exists()]
if missing_checkpoints:
    raise FileNotFoundError("Missing per-site 30-minute checkpoints. Run notebook 1 first. First missing IDs: " + str(missing_checkpoints[:10]))
weather = pd.read_csv(WEATHER_PATH, parse_dates=["datetime"]).set_index("datetime").reindex(INDEX_30MIN)
expected_weather_columns = ["air_temperature_in_degrees_c", "relative_humidity_in_percentage", "wind_speed_in_km_h"]
if list(weather.columns) != expected_weather_columns:
    raise ValueError(f"Unexpected weather columns: {list(weather.columns)}")
if weather.isna().any().any():
    raise ValueError("Weather data contains missing values after reindexing")
print(f"Sites available: {len(site_ids)}")
display(weather.head())

## 3. Create Reproducible Sample Design

Groups may overlap across samples. No duplicates within 1/10/100hh groups. The 1000hh level uses bootstrap with replacement.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
dataset_rows = []
membership_rows = []
next_dataset_no = 25
for level in AGGREGATION_LEVELS:
    for sample_no in range(1, SAMPLES_PER_LEVEL + 1):
        dataset_id = f"ds{next_dataset_no}"
        if level < 1000:
            selected = rng.choice(site_ids, size=level, replace=False).tolist()
        else:
            selected = rng.choice(site_ids, size=level, replace=True).tolist()
        counts = pd.Series(selected, name="edp_site_id").value_counts().sort_index()
        for site_id, weight in counts.items():
            membership_rows.append({"dataset_id": dataset_id, "aggregation_level_hh": level, "sample_no": sample_no, "edp_site_id": site_id, "weight": int(weight)})
        dataset_rows.append({"dataset_id": dataset_id, "dataset_no": next_dataset_no, "aggregation_level_hh": level, "sample_no": sample_no, "sample_label": f"{level}hh_sample{sample_no:02d}", "filename": f"{dataset_id}_aedp_{level}hh_sample{sample_no:02d}_30min_with_weather.csv", "unique_households": int(counts.shape[0]), "total_household_weight": int(counts.sum())})
        next_dataset_no += 1
sample_design = pd.DataFrame(dataset_rows)
membership = pd.DataFrame(membership_rows)
if sample_design.shape[0] != 40 or sample_design["dataset_id"].tolist() != [f"ds{i}" for i in range(25, 65)]:
    raise ValueError("Sample design should contain ds25 through ds64")
sample_design.to_csv(AUDIT_DIR / "aedp_aggregation_sample_design.csv", index=False)
membership.to_csv(AUDIT_DIR / "aedp_aggregation_sample_membership.csv", index=False)
sample_design.to_csv(RESULTS_DIR / "aedp_aggregation_sample_design.csv", index=False)
membership.to_csv(RESULTS_DIR / "aedp_aggregation_sample_membership.csv", index=False)
display(sample_design)

## 4. Load Weighted Site Groups And Export Datasets

In [ ]:
def load_weighted_group_series(weights: pd.Series) -> pd.Series:
    total = pd.Series(0.0, index=INDEX_30MIN, name="netload_kW")
    for site_id, weight in weights.items():
        path = SITE_30MIN_DIR / f"{site_id}_30min.parquet"
        frame = pd.read_parquet(path)
        series = frame.set_index("datetime")["netload_kW"].reindex(INDEX_30MIN)
        if series.isna().any():
            raise ValueError(f"{site_id}: checkpoint has missing values after reindexing")
        total = total.add(series * int(weight), fill_value=0.0)
    return total
export_rows = []
for row in sample_design.itertuples(index=False):
    member = membership.loc[membership["dataset_id"].eq(row.dataset_id)]
    weights = member.set_index("edp_site_id")["weight"]
    netload = load_weighted_group_series(weights)
    dataset = netload.to_frame().merge(weather, left_index=True, right_index=True, how="left").reset_index()
    if dataset.shape[0] != 52608 or dataset.isna().any().any() or dataset["datetime"].duplicated().any():
        raise ValueError(f"{row.dataset_id}: exported dataset failed validation")
    for target_dir in [ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
        out_path = target_dir / row.filename
        dataset.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")
    export_rows.append({"dataset_id": row.dataset_id, "filename": row.filename, "aggregation_level_hh": row.aggregation_level_hh, "sample_no": row.sample_no, "rows": dataset.shape[0], "start": dataset["datetime"].min(), "end": dataset["datetime"].max(), "unique_households": row.unique_households, "total_household_weight": row.total_household_weight})
export_summary = pd.DataFrame(export_rows)
export_summary.to_csv(AUDIT_DIR / "aedp_aggregation_dataset_export_summary.csv", index=False)
export_summary.to_csv(RESULTS_DIR / "aedp_aggregation_dataset_export_summary.csv", index=False)
display(export_summary)

## 5. Debug Notes

If checkpoint files are missing, run notebook 1 first. Re-running overwrites CSV outputs deterministically.